In [ ]:
# @title <b><font color='orange'>WebUI Installer</font></b> {"display-mode":"form"}
# @markdown <a href="https://github.com/gutris1/segsmaker"><img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/></a><br>

Webui = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown --- 
# @markdown * get your civitai api key from [here](https://civitai.com/user/account)
# @markdown * get your huggingface token from [here](https://huggingface.co/settings/tokens)
Civitai___Key = '' # @param { type: "string", placeholder: "Your Civitai API Key (required)" }
HF_Read_Token = '' # @param { type: "string", placeholder: "Your Huggingface READ Token (optional)" }
Mount__GDrive = 'No' # @param ["Yes", "No"]

mount = Mount__GDrive

if mount == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

!curl -sLo /content/setup.py https://github.com/gutris1/segsmaker/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai___Key" --hf_read_token="$HF_Read_Token"

if mount == 'Yes':
    from pathlib import Path
    d = Path('/content/drive/MyDrive/Segsmaker')
    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        f = d / n
        f.mkdir(parents=True, exist_ok=True)
        s = p / f'drive-{n}'
        s.symlink_to(f, target_is_directory=True)
    !rm -rf $WebUI_Output
    o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    o.mkdir(parents=True, exist_ok=True)
    WebUI_Output.symlink_to(o, target_is_directory=True)
    if Webui not in {'ComfyUI', 'SwarmUI'}:
        wc = WebUI / 'cache'
        !rm -rf $wc
        c = d / 'cache'
        c.mkdir(parents=True, exist_ok=True)
        wc.symlink_to(c, target_is_directory=True)


In [ ]:
# @title <b><font color='blue'>Model Downloader - 5 Checkpoint + 5 LoRA + VAE</font></b> {"display-mode":"form"}
Checkpoint_1 = "" # @param {type:"string"}
Checkpoint_2 = "" # @param {type:"string"}
Checkpoint_3 = "" # @param {type:"string"}
Checkpoint_4 = "" # @param {type:"string"}
Checkpoint_5 = "" # @param {type:"string"}
# @markdown ---
Lora_1 = "" # @param {type:"string"}
Lora_2 = "" # @param {type:"string"}
Lora_3 = "" # @param {type:"string"}
Lora_4 = "" # @param {type:"string"}
Lora_5 = "" # @param {type:"string"}
# @markdown ---
VAE_URL = "" # @param {type:"string", placeholder: "URL or leave empty"}
# @markdown ---
Load_from_Drive = False # @param {type:"boolean"}
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:5, step:1}

import concurrent.futures
import subprocess
import shlex
import os

def run_parallel_download(items_with_dirs, max_workers, parallel=True):
    if not items_with_dirs:
        return
    script_template = '''
import sys
import os
sys.path.append("/root/.ipython/profile_default/startup")
try:
    import nenen88
    os.chdir("{target_dir}")
    nenen88.download("{url}")
except Exception as e:
    print(f"Error downloading {url}: {e}")
'''
    if parallel:
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = []
            for url, tdir in items_with_dirs:
                if url.strip():
                    script = script_template.format(target_dir=str(tdir), url=url.strip())
                    futures.append(executor.submit(subprocess.run, [sys.executable, "-c", script]))
            concurrent.futures.wait(futures)
    else:
        for url, tdir in items_with_dirs:
            if url.strip():
                script = script_template.format(target_dir=str(tdir), url=url.strip())
                subprocess.run([sys.executable, "-c", script])

ckpts = [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]
loras = [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]

items = []
for c in ckpts:
    if c.strip(): items.append((c, CKPT))
for l in loras:
    if l.strip(): items.append((l, LORA))
if VAE_URL.strip():
    items.append((VAE_URL, VAE))

if Load_from_Drive:
    print("Load_from_Drive is checked, checking models loaded from Google Drive if mounted.")
run_parallel_download(items, Max_Workers, Parallel_Download)


In [ ]:
# @title <b><font color='green'>Extra Assets - Extensions, Embeddings, Upscalers</font></b> {"display-mode":"form"}
Extension_1 = "" # @param {type:"string", placeholder: "git clone URL or leave empty"}
Extension_2 = "" # @param {type:"string"}
Extension_3 = "" # @param {type:"string"}
Extension_4 = "" # @param {type:"string"}
Extension_5 = "" # @param {type:"string"}
# @markdown ---
Embedding_1 = "" # @param {type:"string"}
Embedding_2 = "" # @param {type:"string"}
Embedding_3 = "" # @param {type:"string"}
# @markdown ---
Upscaler_1 = "" # @param {type:"string"}
Upscaler_2 = "" # @param {type:"string"}
Upscaler_3 = "" # @param {type:"string"}
# @markdown ---
Assets_Parallel_Download = True # @param {type:"boolean"}
Assets_Max_Workers = 3 # @param {type:"slider", min:1, max:5, step:1}

import sys

exts = [Extension_1, Extension_2, Extension_3, Extension_4, Extension_5]
embs = [Embedding_1, Embedding_2, Embedding_3]
upsc = [Upscaler_1, Upscaler_2, Upscaler_3]

items = []
for e in embs:
    if e.strip(): items.append((e, Embeddings))
for u in upsc:
    if u.strip(): items.append((u, Upscalers))

run_parallel_download(items, Assets_Max_Workers, Assets_Parallel_Download)

for ext in exts:
    ext = ext.strip()
    if ext:
        print(f"Cloning extension: {ext}")
        if 'git clone' not in ext: ext = f'git clone {ext}'
        script = f'''
import sys
import os
sys.path.append("/root/.ipython/profile_default/startup")
try:
    import nenen88
    os.chdir("{Extensions}")
    nenen88.clone("{ext}")
except Exception as e:
    print(f"Error cloning {ext}: e")
'''
        subprocess.run([sys.executable, "-c", script])


In [ ]:
# @title <b><font color='purple'>FLUX Model Downloader</font></b> {"display-mode":"form"}
FLUX_Variant = "FLUX.1-schnell (Fast, 4-step)" # @param ["FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
# @markdown ---
FLUX_Unet = "" # @param {type:"string"}
FLUX_Clip_L = "" # @param {type:"string"}
FLUX_T5XXL = "" # @param {type:"string"}
FLUX_VAE = "" # @param {type:"string"}
# @markdown ---
Parallel_FLUX_Download = True # @param {type:"boolean"}
FLUX_Max_Workers = 2 # @param {type:"slider", min:1, max:5, step:1}

items = []
if FLUX_Unet.strip(): items.append((FLUX_Unet, UNET))
if FLUX_Clip_L.strip(): items.append((FLUX_Clip_L, CLIP))
if FLUX_T5XXL.strip(): items.append((FLUX_T5XXL, CLIP))
if FLUX_VAE.strip(): items.append((FLUX_VAE, VAE))

# Auto download based on variant if inputs are empty and it's Forge/ComfyUI (Optional convenience)
if not any([FLUX_Unet.strip(), FLUX_Clip_L.strip(), FLUX_T5XXL.strip(), FLUX_VAE.strip()]):
    print(f"No custom URLs provided. You can manually input them if needed.")

run_parallel_download(items, FLUX_Max_Workers, Parallel_FLUX_Download)


In [ ]:
# @title <b><font color='yellow'>ControlNet Downloader Widget</font></b>
''' Controlnet '''
%run $Controlnet_Widget

In [ ]:
# @title <b><font color='red'>Launcher WebUI</font></b> {"display-mode":"form"}
# @markdown Select the same WebUI that you installed in the first cell.
Software = "Forge-Neo" # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown **Tunnel Tokens (Optional)**
Ngrok_Token = "" # @param {type:"string"}
Zrok_Token = "" # @param {type:"string"}
# @markdown **Extra Settings**
Extra_Args = "" # @param {type:"string"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}
Skip_Widget = False # @param {type:"boolean"}

args_list = []
if Software == "A1111": args_list.append("--xformers")
elif Software == "Forge": args_list.extend(["--disable-xformers", "--opt-sdp-attention", "--cuda-stream"])
elif Software == "ReForge": args_list.extend(["--xformers", "--cuda-stream"])
elif Software == "Forge-Classic": args_list.extend(["--xformers", "--cuda-stream", "--persistent-patches"])
elif Software == "Forge-Neo": args_list.extend(["--xformers", "--cuda-malloc", "--cuda-stream"])
elif Software == "ComfyUI": args_list.extend(["--dont-print-server", "--use-pytorch-cross-attention"])
elif Software == "SwarmUI": args_list.extend(["--launch_mode", "none"])

if Ngrok_Token.strip(): args_list.append(f"--N={Ngrok_Token.strip()}")
if Zrok_Token.strip(): args_list.append(f"--Z={Zrok_Token.strip()}")
if Skip_ComfyUI_Check and Software == "ComfyUI": args_list.append("--skip-comfyui-check")
if Skip_Widget: args_list.append("--skip-widget")
if Extra_Args.strip(): args_list.extend(Extra_Args.strip().split())

cmd_args = " ".join(args_list)
print(f"Launching {Software} with args: {cmd_args}")

%cd -q $WebUI
get_ipython().run_line_magic('run', f"segsmaker.py {cmd_args}")
